# 01 - Data Generation

Generate Plummer sphere mock data, load HDF5 files, and visualize phase-space distributions.

## Prerequisites
```bash
pip install -e ".[notebook]"  # from project root
```

In [ ]:
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

from dpjax.paths import PROJECT_ROOT, DATA_DIR, ensure_dir

# Add scripts/ to path for toy_systems import
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")

## 1. Generate Plummer Sphere Data

In [ ]:
from plummer.plummer_gendata import sample_df, save_data, calc_ideal_loss

N_SAMPLES = 131072  # 2^17

eta = sample_df(N_SAMPLES, max_dist=10.0)
print(f"Generated eta shape: {eta.shape}")
print(f"Ideal loss: {calc_ideal_loss(eta):.6f}")

In [ ]:
# Save to HDF5
ensure_dir(DATA_DIR)
out_path = DATA_DIR / "plummer_n131072.h5"
save_data(eta, str(out_path))
print(f"Saved to {out_path}")

## 2. Load and Inspect Data

In [ ]:
from dpjax.data import load_eta_h5, fit_normalizer

eta = load_eta_h5(out_path)
print(f"Loaded eta: shape={eta.shape}, dtype={eta.dtype}")
print(f"  x  range: [{eta[:, 0].min():.3f}, {eta[:, 0].max():.3f}]")
print(f"  vx range: [{eta[:, 3].min():.3f}, {eta[:, 3].max():.3f}]")

In [ ]:
normalizer = fit_normalizer(eta)
print(f"mean: {normalizer.mean}")
print(f"std:  {normalizer.std}")

eta_std = normalizer.transform(eta)
print(f"\nAfter normalization:")
print(f"  mean ~ {eta_std.mean(axis=0)}")
print(f"  std  ~ {eta_std.std(axis=0)}")

## 3. Visualize Phase-Space Distributions

In [ ]:
labels = ["x", "y", "z", "vx", "vy", "vz"]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for i, (ax, lbl) in enumerate(zip(axes.flat, labels)):
    ax.hist(eta[:, i], bins=100, density=True, alpha=0.7)
    ax.set_xlabel(lbl)
    ax.set_ylabel("density")
fig.suptitle("1D Marginal Distributions (Physical Units)", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# 2D projections: (x, y), (x, vx), (r, |v|)
r = np.sqrt(eta[:, 0]**2 + eta[:, 1]**2 + eta[:, 2]**2)
v = np.sqrt(eta[:, 3]**2 + eta[:, 4]**2 + eta[:, 5]**2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].hist2d(eta[:, 0], eta[:, 1], bins=128, cmap="viridis", density=True)
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[0].set_title("Spatial projection (x, y)")
axes[0].set_aspect("equal")

axes[1].hist2d(eta[:, 0], eta[:, 3], bins=128, cmap="viridis", density=True)
axes[1].set_xlabel("x"); axes[1].set_ylabel("vx")
axes[1].set_title("Phase-space (x, vx)")

axes[2].hist2d(r, v, bins=128, cmap="viridis", density=True)
axes[2].set_xlabel("r"); axes[2].set_ylabel("|v|")
axes[2].set_title("(r, |v|)")

fig.tight_layout()
plt.show()

## 4. Verify Plummer Analytical Profile

The Plummer density profile: $\rho(r) = \frac{3}{4\pi} (1 + r^2)^{-5/2}$

In [ ]:
r_bins = np.linspace(0, 8, 80)
r_mid = 0.5 * (r_bins[:-1] + r_bins[1:])
dr = r_bins[1] - r_bins[0]

counts, _ = np.histogram(r, bins=r_bins)
shell_vol = 4.0 / 3.0 * np.pi * (r_bins[1:]**3 - r_bins[:-1]**3)
rho_empirical = counts / (shell_vol * len(r))

rho_analytic = 3.0 / (4.0 * np.pi) * (1 + r_mid**2)**(-2.5)

plt.figure(figsize=(7, 4.5))
plt.plot(r_mid, rho_analytic, "k-", lw=2, label="Plummer analytic")
plt.plot(r_mid, rho_empirical, "o", ms=3, alpha=0.6, label=f"Sampled (N={N_SAMPLES})")
plt.yscale("log")
plt.xlabel("r")
plt.ylabel(r"$\rho(r)$")
plt.legend()
plt.title("Density Profile")
plt.tight_layout()
plt.show()